# **Задание №4. Разработка спецификации требований к информационной системе (SRS)**

# СПЕЦИФИКАЦИЯ ТРЕБОВАНИЙ К ПРОГРАММНОМУ ОБЕСПЕЧЕНИЮ

## 1. ВВЕДЕНИЕ

### 1.1. Цель документа

Настоящий документ определяет требования к информационной системе автоматизированной сегментации данных дистанционного зондирования Земли (далее — Система), использующей нейросетевую архитектуру HyenaPixel с глобальным рецептивным полем, неявным моделированием фильтров, а также методы самоконтролируемого обучения и оптимизации инференса.
Документ служит основой для последующих этапов проектирования, разработки, тестирования и приемки Системы в рамках учебного проекта.

### 1.2. Область применения

Система предназначена для исследовательских и прикладных задач семантической сегментации спутниковых и аэрофотоснимков, включая классификацию типов землепользования, выделение застройки, дорог, водных объектов и других объектов инфраструктуры.  
Основные пользователи — специалисты по данным ДЗЗ, инженеры по машинному обучению и исследователи, выполняющие эксперименты с архитектурой HyenaPixel и сопоставляющие её с классическими CNN/Transformer‑подходами.

### 1.3. Определения, акронимы и сокращения

- ДЗЗ — данные дистанционного зондирования Земли (спутниковые, аэрофотоснимки и др.).  
- Сегментация — пиксельная классификация изображения на заранее заданные классы.  
- HyenaPixel — нейросетевая архитектура на базе оператора Hyena для обработки изображений с глобальным рецептивным полем и неявным моделированием длинных свёрточных фильтров.
- Самоконтролируемое обучение (Self‑Supervised Learning, SSL) — класс методов обучения без явной разметки, где модель решает вспомогательные задачи, формируя полезные представления.  
- Инференс — процесс применения обученной модели к новым данным для получения результатов сегментации.  
- SRS — Software Requirements Specification, спецификация требований к программному обеспечению.

### 1.4. Пользовательские роли

**Роль 1: Инженер по машинному обучению (ML‑разработчик)**  
Описание: специалист, разрабатывающий и обучающий модели сегментации на основе HyenaPixel и альтернативных архитектур.  
Технический уровень: высокий.  
Основные задачи: конфигурация экспериментов, запуск самоконтролируемого и контролируемого обучения, анализ метрик, экспорт обученных моделей.

**Роль 2: Аналитик ДЗЗ / предметный эксперт**  
Описание: специалист, работающий с тематическими картами и сегментационными масками для решения прикладных задач (землепользование, мониторинг изменений и т.д.).  
Технический уровень: средний.  
Основные задачи: загрузка сцен ДЗЗ, запуск инференса, визуальный анализ результатов, выгрузка сегментационных карт в ГИС‑совместимых форматах.

**Роль 3: Администратор системы**  
Описание: технический специалист, отвечающий за развертывание, настройку и поддержку Системы.  
Технический уровень: высокий.  
Основные задачи: управление пользователями и правами, настройка вычислительной инфраструктуры (GPU/CPU), мониторинг производительности и ресурсов.

### 1.5. Обзор документа

Документ содержит описание функциональных и нефункциональных требований, требований к интерфейсам, ограничений и допущений, а также критериев приемки, структурированных согласно типовой схеме SRS, основанной на ISO/IEC/IEEE 29148 и методических рекомендациях.

## 2. ФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 2.1. Управление данными ДЗЗ

#### ФТ-001: Импорт сцен ДЗЗ

Описание: Система должна обеспечивать загрузку сцен ДЗЗ из локальных файлов и удалённых хранилищ (HTTP(S)/S3‑совместимые источники).  
Входные данные: путь к файлу или URL, параметры авторизации (при необходимости).  
Выходные данные: запись о сцене в каталоге данных Системы, предпросмотр (низкое разрешение), базовые метаданные (пространственное разрешение, размеры, количество каналов).  
Бизнес-правила: поддерживаются форматы GeoTIFF и Cloud‑Optimized GeoTIFF; максимальный размер одной сцены — 20 ГБ.  
Приоритет: Критический.  
Критерий проверки: успешно импортировано не менее 95% корректных файлов тестового набора, метаданные и предпросмотр отображаются без ошибок.

#### ФТ-002: Управление наборами данных и разметкой

Описание: Система должна позволять создавать датасеты, объединяющие несколько сцен и соответствующую разметку (маски сегментации).  
Входные данные: список сцен, путь к файлам разметки, схема классов.  
Выходные данные: описанный датасет с привязкой разметки к сценам и классами.  
Бизнес-правила: разметка должна совпадать по пространственному разрешению и размерам с соответствующей сценой либо должна быть автоматически ресемплирована при импортe.  
Приоритет: Критический.  
Критерий проверки: датасет создаётся, корректно отображается, все пары «изображение‑маска» успешно проверяются на совпадение размеров.

#### ФТ-003: Генерация тайлов для обучения и инференса

Описание: Система должна автоматически нарезать большие сцены на тайлы фиксированного размера для обучения и инференса.  
Входные данные: сцена или датасет, размер тайла (например, 512×512 или 1024×1024), шаг скольжения.  
Выходные данные: набор тайлов изображений и масок (для обучающих датасетов), метаданные о расположении тайлов в исходной сцене.  
Бизнес-правила: тайлы, полностью состоящие из фонового класса (по маске), могут быть по выбору исключены из обучающей выборки.  
Приоритет: Высокий.  
Критерий проверки: нарезка выполняется для тестового набора сцен, количество и покрытие тайлов соответствует параметрам конфигурации.

***

### 2.2. Обучение и самоконтролируемое обучение

#### ФТ-004: Настройка конфигурации модели на основе HyenaPixel

Описание: Система должна предоставлять интерфейс для конфигурации параметров модели HyenaPixel (глубина, ширина, размер фильтра, параметры оператора Hyena и т.п.).
Входные данные: набор гиперпараметров модели и обучения (архитектура, размер фильтра, шаг обучения, размер батча и т.д.).  
Выходные данные: сохранённый профиль конфигурации эксперимента.  
Бизнес-правила: конфигурация должна проходить валидацию (диапазоны значений, совместимость параметров).  
Приоритет: Критический.  
Критерий проверки: инженер ML может создать, сохранить и повторно использовать конфигурацию для запуска обучения.

#### ФТ-005: Самоконтролируемое предварительное обучение

Описание: Система должна поддерживать запуск самоконтролируемого обучения представлений на неразмеченных данных ДЗЗ (например, маскирование участков, предсказание трансформаций, контрастивное обучение).  
Входные данные: список датасетов с неразмеченными сценами, конфигурация SSL‑задачи, профиль модели.  
Выходные данные: чекпоинт предварительно обученной модели, лог обучения, значения SSL‑метрик (loss, proxy‑метрики).  
Бизнес-правила: SSL‑задача должна быть совместима с размером и количеством каналов входных данных.  
Приоритет: Высокий.  
Критерий проверки: предварительное обучение завершается без ошибок, чекпоинт успешно используется в последующем контролируемом обучении.

#### ФТ-006: Контролируемое обучение для сегментации

Описание: Система должна обеспечивать запуск контролируемого обучения модели сегментации на размеченных датасетах, включая возможность дообучения от SSL‑чекпоинта.  
Входные данные: датасет с разметкой, конфигурация модели, параметры обучения (эпохи, scheduler, аугментации).  
Выходные данные: обученная модель, история метрик (loss, IoU, F1 по классам), лучшие чекпоинты по выбранному критерию.  
Бизнес-правила: поддерживается ранняя остановка по валидационному набору; лог обучения должен быть доступен для последующего анализа.  
Приоритет: Критический.  
Критерий проверки: обучение на демонстрационном датасете завершается, достигается заданный порог IoU (см. критерии приемки), чекпоинт проходит тестовый инференс.

***

### 2.3. Инференс и анализ результатов

#### ФТ-007: Инференс на отдельных сценах и батчах сцен

Описание: Система должна выполнять инференс обученной модели на отдельных сценах и батчах сцен ДЗЗ, включая автоматическую нарезку и склейку тайлов.  
Входные данные: выбранная модель, список сцен, параметры инференса (размер тайла, перекрытие, порог по вероятности).  
Выходные данные: карты сегментации в виде rasters/масок, метаданные о времени обработки и потреблённых ресурсах.  
Бизнес-правила: при батчевом инференсе должен обеспечиваться контроль максимальной загрузки GPU/CPU (например, через размер батча тайлов).  
Приоритет: Критический.  
Критерий проверки: инференс на тестовом наборе сцен выполняется без ошибок, генерируются корректные маски нужных размеров, время обработки логируется.

#### ФТ-008: Визуализация и интерактивный просмотр результатов

Описание: Система должна предоставлять пользователю возможность визуализировать исходные сцены, наложенные карты сегментации и, опционально, карты неопределённости/уверенности модели.  
Входные данные: сцена, соответствующая карта сегментации, (опционально) карты вероятностей по классам.  
Выходные данные: интерактивное представление (панорама, зум, переключение слоёв и классов).  
Бизнес-правила: визуализация должна корректно отображать геопривязку сцен при наличии координат.  
Приоритет: Высокий.  
Критерий проверки: аналитик ДЗЗ может просмотреть результаты хотя бы для трёх сцен, переключая слои и масштабы без заметных задержек.

#### ФТ-009: Оценка качества на тестовых наборах

Описание: Система должна рассчитывать метрики качества сегментации (IoU, F1, Accuracy и др.) на тестовых датасетах с разметкой.  
Входные данные: выбранная модель, тестовый датасет с разметкой, список метрик.  
Выходные данные: значения метрик по классам и в среднем, отчёт в машинно‑читаемом (JSON/CSV) и человекочитаемом форматах.  
Бизнес-правила: для несбалансированных классов должна быть возможность вычислять взвешенные и макро‑усреднённые метрики.  
Приоритет: Критический.  
Критерий проверки: результаты метрик устойчивы к повторным запускам (при фиксированном seed), отчёты корректно сохраняются и доступны для анализа.

***

### 2.4. Администрирование и мониторинг

#### ФТ-010: Управление пользователями и ролями

Описание: Система должна поддерживать создание, удаление и редактирование учётных записей пользователей, а также назначение ролей (ML‑разработчик, Аналитик, Администратор).  
Входные данные: данные пользователя (ФИО, email/логин), выбранная роль.  
Выходные данные: созданная или обновлённая учётная запись.  
Бизнес-правила: только Администратор может изменять роли; учётная запись должна быть уникальна по логину/email.  
Приоритет: Средний.  
Критерий проверки: Администратор может создать пользователя каждой роли и выполнить тестовый вход под соответствующими учётными данными.

#### ФТ-011: Мониторинг экспериментов и загрузки ресурсов

Описание: Система должна предоставлять обзор запущенных и завершённых задач обучения/инференса и показатели загрузки ключевых ресурсов (GPU, CPU, память).  
Входные данные: запрос на просмотр состояния Системы.  
Выходные данные: список задач с статусами (в очереди/выполняется/завершено/ошибка) и агрегированные показатели загрузки ресурсов.  
Бизнес-правила: доступ к детализированным данным о ресурсах может быть ограничен только для Администратора.  
Приоритет: Средний.  
Критерий проверки: Администратор видит в интерфейсе список задач и корректные показатели загрузки при искусственной нагрузке.

#### ФТ-012: Экспорт моделей и результатов

Описание: Система должна позволять экспортировать обученные модели и результаты инференса в стандартные форматы для дальнейшего использования.  
Входные данные: идентификатор модели или набора результатов, требуемый формат экспорта.  
Выходные данные: файл модели (например, ONNX/PyTorch) и/или сегментационные карты (GeoTIFF/PNG) для загрузки или передачи во внешние системы.  
Бизнес-правила: при экспорте сегментационных карт сохраняется геопривязка (если она есть в исходной сцене).  
Приоритет: Высокий.  
Критерий проверки: экспортированные модели и карты успешно загружаются в внешние инструменты (фреймворк ML, ГИС).


## 3. НЕФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 3.1. Требования к производительности

**НФТ-П-001: Время инференса на тайле**  
Система должна обеспечивать среднее время инференса не более 0,5 секунды на один тайл размером 1024×1024 пикселей при использовании GPU уровня NVIDIA RTX 3060 или выше при батче из не менее чем 4 тайлов.

**НФТ-П-002: Время обучения базовой модели**  
Полный проход по обучающей выборке (одна эпоха) для датасета объёмом 10 000 тайлов 512×512 не должен превышать 20 минут на одном GPU уровня NVIDIA RTX 3060 при стандартной конфигурации модели.

**НФТ-П-003: Время отклика пользовательского интерфейса**  
Время отображения страницы списка экспериментов и результатов не должно превышать 2 секунд при количестве записей до 500.

### 3.2. Требования к надежности и доступности

**НФТ-Н-001: Доступность системы**  
Система должна быть доступна не менее 95% времени в течение календарного месяца в условиях лабораторной инфраструктуры.

**НФТ-Н-002: Сохранность результатов обучения и инференса**  
Чекпоинты моделей и результаты инференса должны сохраняться на устойчивом к сбоям хранилище; потеря данных при отказе сервера не должна превышать результаты одного последнего запущенного задания.

**НФТ-Н-003: Обработка ошибок задач**  
В случае ошибок при обучении или инференсе Система должна корректно завершать задачу, сохранять лог и отображать понятное сообщение пользователю.

### 3.3. Требования к безопасности

**НФТ-Б-001: Аутентификация и авторизация**  
Доступ к Системе должен осуществляться только по именам пользователей и паролям; изменение конфигураций и запуск обучения/инференса должны быть доступны только аутентифицированным пользователям.

**НФТ-Б-002: Разграничение прав доступа**  
Роли пользователей должны определять набор доступных функций: Аналитик не может запускать обучение моделей, Администратор может управлять пользователями и конфигурацией.

**НФТ-Б-003: Защита данных**  
При использовании сетевого доступа к интерфейсу Системы передача данных должна осуществляться по протоколу HTTPS; конфиденциальные данные доступа к внешним хранилищам (например, S3‑ключи) должны храниться в зашифрованном виде.

### 3.4. Требования к удобству использования

**НФТ-Ю-001: Обучаемость**  
Новый пользователь — аналитик ДЗЗ — должен иметь возможность загрузить сцену и выполнить инференс с использованием уже обученной модели в течение 30 минут после краткого ознакомления с интерфейсом без обращения к внешней документации.

**НФТ-Ю-002: Интуитивность интерфейса**  
Структура интерфейса должна следовать типовым паттернам веб‑приложений: доступ к датасетам, моделям, задачам и результатам осуществляется через очевидные разделы меню.

**НФТ-Ю-003: Визуальная обратная связь**  
Все долгие операции (обучение, инференс) должны сопровождаться индикатором прогресса и статусами задач.

### 3.5. Требования к совместимости и переносимости

**НФТ-С-001: Поддерживаемые браузеры**  
Веб‑интерфейс Системы должен корректно работать в актуальных версиях браузеров Google Chrome, Mozilla Firefox и Yandex Browser.

**НФТ-С-002: Совместимость форматов данных**  
Система должна поддерживать чтение и запись GeoTIFF‑файлов, совместимых с распространёнными ГИС‑системами (QGIS, ArcGIS).

### 3.6. Требования к масштабируемости

**НФТ-М-001: Масштабирование по данным**  
Архитектура Системы должна позволять обрабатывать увеличивающиеся объёмы данных (до 100 000 тайлов в датасете) без изменения архитектурных решений, за счёт добавления вычислительных ресурсов.

**НФТ-М-002: Масштабирование по пользователям**  
Система должна поддерживать одновременную работу не менее 5 пользователей без существенной деградации производительности интерфейса и очереди задач.

### 3.7. Специфические требования к ML‑модели

**НФТ-МО-001: Качество сегментации**  
Модель HyenaPixel после обучения должна обеспечивать средний IoU по всем классам не менее 0,6 на тестовом датасете, определённом в проекте.

**НФТ-МО-002: Влияние предварительного обучения**  
Использование самоконтролируемого предварительного обучения должно давать прирост среднего IoU не менее 0,03 по сравнению с моделью, обученной «с нуля» при прочих равных условиях (по результатам абляционного исследования).

**НФТ-МО-003: Стабильность инференса**  
Результаты сегментации не должны существенно изменяться при небольших вариациях освещённости и шума; изменение IoU при таких аугментациях не должно превышать 0,05 на тестовом наборе.


## 4. ТРЕБОВАНИЯ К ИНТЕРФЕЙСАМ

### 4.1. Пользовательский интерфейс

**ТИ-ПИ-001: Главная страница / Дашборд**  
Должна отображать список последних экспериментов (обучение, инференс), их статус и основные метрики, а также быстрые ссылки на разделы «Датасеты», «Модели», «Задачи».

**ТИ-ПИ-002: Экран управления датасетами**  
Должен предоставлять возможность импорта сцен, создания датасетов, просмотра списка сцен и их метаданных, запуска нарезки на тайлы.

**ТИ-ПИ-003: Экран эксперимента обучения**  
Должен отображать конфигурацию модели и обучения, текущий прогресс (номер эпохи, значения метрик), логи и кнопки управления (остановка/продолжение при наличии).

**ТИ-ПИ-004: Экран инференса и визуализации**  
Должен позволять выбрать модель и сцены, запустить инференс и затем визуально просмотреть результат с наложением маски, легендой классов и возможностью экспорта.

### 4.2. Программный интерфейс (API)

**ТИ-АПИ-001: REST API для инференса**  
Система должна предоставлять REST‑endpoint, принимающий сцены или тайлы и возвращающий сегментационные карты (например, в виде GeoTIFF или массива классов) с указанием версии модели, использованной для инференса.

**ТИ-АПИ-002: REST API для управления задачами**  
Должен быть предусмотрен API для создания задач обучения и инференса, получения их статусов и результатов, что позволяет автоматизировать экспериментальные сценарии.

### 4.3. Интерфейсы с внешними системами

**ТИ-ВС-001: Интеграция с хранилищами данных**  
Система должна поддерживать подключение к внешним объектным хранилищам (например, S3‑совместимым) для чтения и записи сцен ДЗЗ и результатов.

**ТИ-ВС-002: Интеграция с ГИС‑инструментами**  
Экспортированные карты сегментации должны быть совместимы с внешними ГИС‑платформами, чтобы аналитик ДЗЗ мог продолжать обработку вне Системы.


## 5. ОГРАНИЧЕНИЯ И ДОПУЩЕНИЯ

### 5.1. Ограничения

**ОГР-001: Вычислительные ресурсы**  
Разработка Системы осуществляется в рамках учебного проекта и ограничена доступностью 1–2 GPU среднего уровня и ограниченным объёмом дискового пространства.

**ОГР-002: Объём и качество данных**  
Обучение и оценка Системы выполняются на ограниченном количестве открытых датасетов ДЗЗ, что может влиять на обобщающую способность модели.

**ОГР-003: Временные рамки**  
Срок разработки ограничен академическим периодом, что накладывает ограничения на глубину оптимизации и объём реализуемой функциональности.

### 5.2. Допущения

**ДОП-001: Доступность открытых данных ДЗЗ**  
Предполагается наличие достаточного количества открытых сцен ДЗЗ и, по возможности, разметки для решения выбранной задачи сегментации.

**ДОП-002: Наличие базовой инфраструктуры**  
Предполагается наличие инфраструктуры (сервер или рабочая станция) с установленными GPU‑драйверами и необходимым программным стеком (Python, ML‑фреймворк, веб‑сервер).

**ДОП-003: Уровень подготовки пользователей**  
Предполагается, что ML‑разработчики и аналитики ДЗЗ обладают базовыми навыками работы с веб‑интерфейсами и пониманием предметной области.


## 6. КРИТЕРИИ ПРИЕМКИ

### 6.1. Функциональная приемка

**КП-Ф-001:** Все функциональные требования с приоритетом «Критический» должны быть реализованы и успешно протестированы на демонстрационном наборе данных.  
**КП-Ф-002:** Полный пользовательский сценарий «Импорт сцены → создание датасета → обучение модели → инференс → экспорт результатов» выполняется без ошибок.  
**КП-Ф-003:** Пользователь‑аналитик может без участия разработчика выполнить инференс уже обученной модели и получить пригодные для анализа карты сегментации.

### 6.2. Нефункциональная приемка

**КП-НФ-001:** Измеренные показатели производительности соответствуют требованиям НФТ-П-001 и НФТ-П-002 на целевой аппаратной конфигурации.  
**КП-НФ-002:** Качество сегментации на тестовом датасете соответствует или превышает порог, заданный в НФТ-МО-001, а эффект SSL‑предобучения соответствует НФТ-МО-002.  
**КП-НФ-003:** Система демонстрирует стабильную работу в течение не менее 8 часов непрерывной эксплуатации без критических отказов.

### 6.3. Документация

**КП-Д-001:** Подготовлено пользовательское руководство, описывающее основные сценарии работы для ролей «ML‑разработчик» и «Аналитик ДЗЗ».  
**КП-Д-002:** Подготовлена краткая техническая документация по архитектуре, конфигурации модели HyenaPixel и процессу развёртывания Системы.

### 6.4. Тестирование

**КП-Т-001:** Проведено модульное тестирование ключевых компонентов (импорт данных, нарезка тайлов, инференс, расчёт метрик) с покрытием кода не менее 70% в этих компонентах.  
**КП-Т-002:** Проведено интеграционное тестирование основных пользовательских сценариев, выявленные дефекты критичности выше «Средний» устранены.  
**КП-Т-003:** Проведено экспериментальное тестирование моделей с фиксированными конфигурациями, результаты зафиксированы в отчётах и воспроизводимы при повторных запусках.